In [10]:
import pandas as pd
import lightgbm as lgb

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")

CAT_COLS = ["main_category", "sub_category", "sub_sub_category", "condition_label"]
NUM_COLS = ["item_condition_id", "shipping", "category_depth", "name_length",
            "desc_length", "name_word_count", "has_description", "is_branded",
            "cat_avg_price", "brand_avg_price"]

for c in CAT_COLS:
    train[c] = train[c].fillna("missing").astype("category")
    val[c] = val[c].fillna("missing").astype("category")

X_train = train[NUM_COLS + CAT_COLS]
X_val = val[NUM_COLS + CAT_COLS]
y_train = train["log_price"]
y_val = val["log_price"]

print(X_train.dtypes)
print("Shape:", X_train.shape, X_val.shape)

item_condition_id       int64
shipping                int64
category_depth          int64
name_length             int64
desc_length             int64
name_word_count         int64
has_description         int64
is_branded              int64
cat_avg_price         float64
brand_avg_price       float64
main_category        category
sub_category         category
sub_sub_category     category
condition_label      category
dtype: object
Shape: (39780, 14) (9945, 14)


In [11]:
import numpy as np
from sklearn.metrics import mean_squared_error

model_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

model_lgb.fit(X_train, y_train)
pred_log_lgb = model_lgb.predict(X_val)
rmsle_lgb = np.sqrt(mean_squared_error(y_val, pred_log_lgb))

print("LightGBM val RMSLE:", round(rmsle_lgb, 4))
print("Ridge val RMSLE", 0.5204)

LightGBM val RMSLE: 0.5677
Ridge val RMSLE 0.5204


In [12]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
import scipy.sparse as sp

scaler_struct = StandardScaler()
X_num_train_struct = scaler_struct.fit_transform(train[NUM_COLS])
X_num_val_struct = scaler_struct.transform(val[NUM_COLS])

ohe_struct = OneHotEncoder(handle_unknown="ignore")
X_cat_train_struct = ohe_struct.fit_transform(train[CAT_COLS])
X_cat_val_struct = ohe_struct.transform(val[CAT_COLS])

X_train_struct = sp.hstack([X_num_train_struct, X_cat_train_struct]).tocsr()
X_val_struct = sp.hstack([X_num_val_struct, X_cat_val_struct]).tocsr()

ridge_struct = Ridge(alpha=0.5, random_state=42)
ridge_struct.fit(X_train_struct, y_train)

pred_log_ridge_struct = ridge_struct.predict(X_val_struct)
rmsle_ridge_struct = np.sqrt(mean_squared_error(y_val, pred_log_ridge_struct))

print("Ridge (structured only, no text)  val RMSLE:", round(rmsle_ridge_struct, 4))
print("LightGBM (structured only)        val RMSLE:", round(rmsle_lgb, 4))
print("Ridge (structured + text)  val RMSLE:", 0.5204)

Ridge (structured only, no text)  val RMSLE: 0.5983
LightGBM (structured only)        val RMSLE: 0.5677
Ridge (structured + text)  val RMSLE: 0.5204


In [13]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error

for c in CAT_COLS:
    train[c] = train[c].fillna("missing").astype("category")
    cat_dtype = pd.CategoricalDtype(categories=train[c].cat.categories)
    val[c] = val[c].fillna("missing").astype(cat_dtype)

X_train = train[NUM_COLS + CAT_COLS]
X_val = val[NUM_COLS + CAT_COLS]

model_xgb = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    enable_categorical=True,
    tree_method="hist"
)
model_xgb.fit(X_train, y_train)
pred_log_xgb = model_xgb.predict(X_val)
rmsle_xgb = np.sqrt(mean_squared_error(y_val, pred_log_xgb))

print("XGBoost (structured only)   val RMSLE:", round(rmsle_xgb, 4))
print("LightGBM (structured only)  val RMSLE:", round(rmsle_lgb, 4))
print("Ridge (structured only)     val RMSLE:", round(rmsle_ridge_struct, 4))

C:\Users\hp\AppData\Local\Temp\ipykernel_1656\3849016265.py:8: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  val[c] = val[c].fillna("missing").astype(cat_dtype)
C:\Users\hp\AppData\Local\Temp\ipykernel_1656\3849016265.py:8: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  val[c] = val[c].fillna("missing").astype(cat_dtype)


XGBoost (structured only)   val RMSLE: 0.5753
LightGBM (structured only)  val RMSLE: 0.5677
Ridge (structured only)     val RMSLE: 0.5983


In [14]:
print("Train shape:", train.shape)
print(train["log_price"].describe())
print("\nX shape:", X.shape)
print("\nCAT_COLS unique counts:")
for c in CAT_COLS:
    print(f"  {c}: {train[c].nunique()} categories")

Train shape: (39780, 22)
count    39780.000000
mean         2.975819
std          0.743853
min          1.386294
25%          2.397895
50%          2.890372
75%          3.401197
max          7.095893
Name: log_price, dtype: float64

X shape: (39780, 14)

CAT_COLS unique counts:
  main_category: 10 categories
  sub_category: 109 categories
  sub_sub_category: 625 categories
  condition_label: 5 categories


In [15]:
import pandas as pd
import numpy as np
import lightgbm as lgb 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error

train = pd.read_csv("../data/processed/train.csv")
CAT_COLS = ["main_category", "sub_category", "sub_sub_category", "condition_label"]
NUM_COLS = ["item_condition_id", "shipping", "category_depth", "name_length",
            "desc_length", "name_word_count", "has_description", "is_branded",
            "cat_avg_price", "brand_avg_price"]
for c in CAT_COLS:
    train[c] = train[c].fillna("missing").astype("category")


X = train[NUM_COLS + CAT_COLS]
y = train["log_price"]
strat_col = train["price_bin"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for fol_num, (tr_idx, va_idx) in enumerate(skf.split(X, strat_col), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, 
                              random_state=42,
                              verbose=-1)

    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    rmsle = np.sqrt(mean_squared_error(y_va, pred))
    fold_scores.append(rmsle)
    print(f"Fold {fol_num}: RMSLE = {rmsle:.4f}")

print(f"\nMean RMSLE: {np.mean(fold_scores):.4f} | Std: {np.std(fold_scores):.4f}")

Fold 1: RMSLE = 0.5667
Fold 2: RMSLE = 0.5655
Fold 3: RMSLE = 0.5663
Fold 4: RMSLE = 0.5650
Fold 5: RMSLE = 0.5702

Mean RMSLE: 0.5667 | Std: 0.0018


In [16]:
import xgboost as xgb

fold_scores_xgb = []

for fold_num, (tr_idx, va_idx) in enumerate(skf.split(X, strat_col), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                              random_state=42, enable_categorical=True, tree_method="hist")
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    rmsle = np.sqrt(mean_squared_error(y_va, pred))
    fold_scores_xgb.append(rmsle)
    print(f"Fold {fold_num}: RMSLE = {rmsle:.4f}")

print(f"\nXGBoost Mean RMSLE: {np.mean(fold_scores_xgb):.4f} | Std: {np.std(fold_scores_xgb):.4f}")
print(f"LightGBM Mean RMSLE: {np.mean(fold_scores):.4f} | Std: {np.std(fold_scores):.4f}")

Fold 1: RMSLE = 0.5813
Fold 2: RMSLE = 0.5776
Fold 3: RMSLE = 0.5827
Fold 4: RMSLE = 0.5754
Fold 5: RMSLE = 0.5816

XGBoost Mean RMSLE: 0.5797 | Std: 0.0027
LightGBM Mean RMSLE: 0.5667 | Std: 0.0018


In [17]:
fold_scores_lgb_es = []
best_iterations = []

for fold_num, (tr_idx, va_idx) in enumerate(skf.split(X, strat_col), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=31,
                               random_state=42, verbose=-1)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    pred = model.predict(X_va)
    rmsle = np.sqrt(mean_squared_error(y_va, pred))
    fold_scores_lgb_es.append(rmsle)
    best_iterations.append(model.best_iteration_)
    print(f"Fold {fold_num}: RMSLE = {rmsle:.4f}  (stopped at {model.best_iteration_} trees)")

print(f"\nWith early stopping - Mean: {np.mean(fold_scores_lgb_es):.4f} | Std: {np.std(fold_scores_lgb_es):.4f}")
print(f"Fixed 500 trees      - Mean: {np.mean(fold_scores):.4f} | Std: {np.std(fold_scores):.4f}")

c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1: RMSLE = 0.5649  (stopped at 110 trees)


c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2: RMSLE = 0.5631  (stopped at 218 trees)


c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 3: RMSLE = 0.5650  (stopped at 135 trees)


c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4: RMSLE = 0.5643  (stopped at 233 trees)


c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5: RMSLE = 0.5696  (stopped at 198 trees)

With early stopping - Mean: 0.5654 | Std: 0.0022
Fixed 500 trees      - Mean: 0.5667 | Std: 0.0018


In [18]:
fold_scores_xgb_es = []

for fold_num, (tr_idx, va_idx) in enumerate(skf.split(X, strat_col), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=6,
                              random_state=42, enable_categorical=True, tree_method="hist",
                              early_stopping_rounds=50, eval_metric="rmse")
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    pred = model.predict(X_va)
    rmsle = np.sqrt(mean_squared_error(y_va, pred))
    fold_scores_xgb_es.append(rmsle)
    print(f"Fold {fold_num}: RMSLE = {rmsle:.4f}  (stopped at {model.best_iteration} trees)")

print(f"\nXGBoost + early stopping - Mean: {np.mean(fold_scores_xgb_es):.4f} | Std: {np.std(fold_scores_xgb_es):.4f}")
print(f"LightGBM + early stopping - Mean: {np.mean(fold_scores_lgb_es):.4f} | Std: {np.std(fold_scores_lgb_es):.4f}")

Fold 1: RMSLE = 0.5704  (stopped at 86 trees)
Fold 2: RMSLE = 0.5674  (stopped at 96 trees)
Fold 3: RMSLE = 0.5716  (stopped at 93 trees)
Fold 4: RMSLE = 0.5675  (stopped at 96 trees)
Fold 5: RMSLE = 0.5723  (stopped at 84 trees)

XGBoost + early stopping - Mean: 0.5699 | Std: 0.0021
LightGBM + early stopping - Mean: 0.5654 | Std: 0.0022


In [19]:
final_lgb = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=31,
                               random_state=42, verbose=-1)
final_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

pred_final = final_lgb.predict(X_val)
rmsle_final = np.sqrt(mean_squared_error(y_val, pred_final))
print("Final LightGBM val RMSLE:", round(rmsle_final, 4), "| trees used:", final_lgb.best_iteration_)

import joblib
joblib.dump(final_lgb, "../models/lightgbm_final.joblib")
print("Saved to models/lightgbm_final.joblib")

c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Final LightGBM val RMSLE: 0.5673 | trees used: 273
Saved to models/lightgbm_final.joblib
